Lookup failed: [Errno 1] Unknown host


In [18]:
#!/usr/bin/env python3
"""
Robust Real-vs-Fake (heuristic) detector using OpenCV + MediaPipe face mesh.

How it works:
- Threaded capture to maximize frame rate.
- Brightness std dev (sliding window).
- Optical flow mean magnitude between frames (motion).
- Sharpness via variance of Laplacian.
- Blink detection using MediaPipe face landmarks -> EAR-like metric.

Decide REAL if:
 - blink detected OR
 - sufficient motion AND good sharpness AND no strong periodic flicker in brightness.

Tune thresholds at top of file for your camera/lighting.
"""
import cv2
import numpy as np
import mediapipe as mp
import time
from collections import deque
from threading import Thread

# ---------------------- TUNE THESE ----------------------
REQUESTED_FPS = 60
WIDTH = 640
HEIGHT = 480

BRIGHTNESS_WINDOW = 60           # how many frames to keep for brightness stats
BRIGHTNESS_STD_FLICKER = 8.0     # stddev threshold above which flicker likely
MOTION_THRESHOLD = 1.5           # low motion < this => suspicious
SHARPNESS_THRESHOLD = 60.0       # Laplacian variance threshold for "detailed" image
BLINK_EAR_THRESHOLD = 0.20       # EAR-like threshold per eye for blink
BLINK_CONSEC_FRAMES = 2          # frames EAR must be below threshold to count as blink
MIN_REAL_SCORE = 1               # number of passing checks to declare REAL
# -------------------------------------------------------

# MediaPipe mesh setup
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False,
                                  max_num_faces=1,
                                  refine_landmarks=True,
                                  min_detection_confidence=0.5,
                                  min_tracking_confidence=0.5)

# Useful Mediapipe indices for eyes (refined landmarks)
# left eye: [33, 160, 158, 133, 153, 144]
# right eye: [263, 387, 385, 362, 380, 373]
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [263, 387, 385, 362, 380, 373]

def eye_aspect_ratio(landmarks, eye_indices, img_w, img_h):
    pts = []
    for idx in eye_indices:
        lm = landmarks[idx]
        pts.append((int(lm.x * img_w), int(lm.y * img_h)))
    # pts order: p0..p5 as in formula
    # compute euclidean distances
    def dist(a, b):
        return np.linalg.norm(np.array(a) - np.array(b))
    A = dist(pts[1], pts[5])
    B = dist(pts[2], pts[4])
    C = dist(pts[0], pts[3])
    if C == 0:
        return 0.0
    ear = (A + B) / (2.0 * C)
    return ear

class ThreadedCam:
    def __init__(self, src=0, width=WIDTH, height=HEIGHT, fps=REQUESTED_FPS):
        self.cap = cv2.VideoCapture(src, cv2.CAP_DSHOW if cv2.__version__ >= '4' and (cv2.getBuildInformation().find('MSVC')!=-1) else src)
        # set properties
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)
        self.cap.set(cv2.CAP_PROP_FPS, fps)
        # fallback if camera doesn't support these
        self.ret, self.frame = self.cap.read()
        self.stopped = False
        self.thread = Thread(target=self.update, daemon=True)
        self.thread.start()

    def update(self):
        while not self.stopped:
            ret, frame = self.cap.read()
            if not ret:
                self.ret = False
                self.frame = None
                continue
            self.ret = True
            self.frame = frame

    def read(self):
        return self.ret, self.frame

    def stop(self):
        self.stopped = True
        time.sleep(0.1)
        self.cap.release()

def main():
    cam = ThreadedCam(src=0)
    time.sleep(0.5)  # warm up
    prev_gray = None
    brightness_hist = deque(maxlen=BRIGHTNESS_WINDOW)
    prev_time = time.time()

    # blink counters
    left_blink_count = 0
    right_blink_count = 0
    left_blink_frames = 0
    right_blink_frames = 0

    print("Starting. Press ESC to quit. Allow a few seconds for MediaPipe warm-up...")

    while True:
        ok, frame = cam.read()
        if not ok or frame is None:
            # no frame available
            time.sleep(0.01)
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        h, w = frame.shape[:2]

        # brightness
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        brightness = float(np.mean(gray))
        brightness_hist.append(brightness)
        brightness_std = float(np.std(brightness_hist)) if len(brightness_hist) > 1 else 0.0

        # sharpness (variance of Laplacian)
        lap = cv2.Laplacian(gray, cv2.CV_64F)
        sharpness = float(lap.var())

        # optical flow (motion)
        motion = 0.0
        if prev_gray is not None:
            # calc optical flow with Farneback
            flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None,
                                                pyr_scale=0.5, levels=3, winsize=15,
                                                iterations=3, poly_n=5, poly_sigma=1.2, flags=0)
            mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
            motion = float(np.mean(mag))
        prev_gray = gray

        # face landmarks (mediapipe)
        results = face_mesh.process(frame_rgb)
        blink_detected = False
        name = "NoFace"
        if results.multi_face_landmarks and len(results.multi_face_landmarks) > 0:
            landmarks = results.multi_face_landmarks[0].landmark
            # compute EAR for both eyes
            left_ear = eye_aspect_ratio(landmarks, LEFT_EYE, w, h)
            right_ear = eye_aspect_ratio(landmarks, RIGHT_EYE, w, h)

            # process left eye frames
            if left_ear < BLINK_EAR_THRESHOLD:
                left_blink_frames += 1
            else:
                if left_blink_frames >= BLINK_CONSEC_FRAMES:
                    left_blink_count += 1
                left_blink_frames = 0

            # process right eye frames
            if right_ear < BLINK_EAR_THRESHOLD:
                right_blink_frames += 1
            else:
                if right_blink_frames >= BLINK_CONSEC_FRAMES:
                    right_blink_count += 1
                right_blink_frames = 0

            # if either eye blinked recently (in the last few seconds)
            if left_blink_count > 0 or right_blink_count > 0:
                blink_detected = True

            name = f"L_EAR:{left_ear:.2f} R_EAR:{right_ear:.2f}"

        # flicker heuristics: high stddev of brightness suggests screen flicker
        flicker = (brightness_std > BRIGHTNESS_STD_FLICKER)

        # Compose a score from checks
        checks = {
            "blink": blink_detected,
            "motion": motion > MOTION_THRESHOLD,
            "sharp": sharpness > SHARPNESS_THRESHOLD,
            "no_flicker": not flicker
        }
        score = sum(1 for v in checks.values() if v)

        # Decision logic:
        # - If blink detected -> real
        # - Else if at least MIN_REAL_SCORE checks true -> real
        # - Else fake
        is_real = False
        if blink_detected:
            is_real = True
        elif score >= MIN_REAL_SCORE:
            is_real = True
        else:
            is_real = False

        # Display diagnostics
        label = "REAL" if is_real else "FAKE"
        color = (0, 255, 0) if is_real else (0, 0, 255)
        cv2.putText(frame, f"Decision: {label}", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
        cv2.putText(frame, f"Motion:{motion:.2f} Sharp:{sharpness:.1f} Bstd:{brightness_std:.2f}", (10, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1)
        cv2.putText(frame, f"Blink:{blink_detected} Lcnt:{left_blink_count} Rcnt:{right_blink_count}", (10, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1)
        cv2.putText(frame, name, (10, 105), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

        # Optionally draw landmarks for debugging
        if results.multi_face_landmarks:
            mp.solutions.drawing_utils.draw_landmarks(
                frame,
                results.multi_face_landmarks[0],
                mp_face_mesh.FACEMESH_TESSELATION,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp.solutions.drawing_styles
                .get_default_face_mesh_tesselation_style())

        cv2.imshow("AntiSpoof (ESC to quit)", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == 27:
            break

    cam.stop()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


2025-11-02 21:37:58.057936: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-02 21:37:58.074084: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-02 21:37:58.185528: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-02 21:37:58.291606: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762099678.409961  277108 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762099678.44

Starting. Press ESC to quit. Allow a few seconds for MediaPipe warm-up...


W0000 00:00:1762099686.989842  287759 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
